# Algoritmo de Grover

## Objetivo
Busca em banco de dados não estruturado com **speedup quadrático**.

## Vantagem Quântica
- **Clássico:** O(N) consultas
- **Quântico:** O(√N) consultas

## Referências
- Livro: Capítulo 9
- [Qiskit Textbook](https://qiskit.org/textbook/ch-algorithms/grover.html)

In [ ]:
# Imports
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
import numpy as np

## 1. Criar o Oráculo de Marcação

O oráculo marca o estado alvo invertendo sua fase.

In [ ]:
def create_oracle(n_qubits: int, target: str) -> QuantumCircuit:
    """
    Cria um oráculo que marca o estado alvo invertendo sua fase.
    
    Args:
        n_qubits: número de qubits
        target: estado alvo em binário (ex: "101")
    
    Returns:
        QuantumCircuit: oráculo
    """
    oracle = QuantumCircuit(n_qubits)
    
    # Converter target para lista de bits (invertida para little-endian)
    target_bits = target[::-1]
    
    # Aplicar X nos qubits que devem ser |0⟩ no target
    for i, bit in enumerate(target_bits):
        if bit == '0':
            oracle.x(i)
    
    # Aplicar MCZ (Multi-Controlled Z) - inverte fase se todos qubits = |1⟩
    # Usando decomposição com H e MCX
    oracle.h(n_qubits - 1)
    oracle.mcx(list(range(n_qubits - 1)), n_qubits - 1)
    oracle.h(n_qubits - 1)
    
    # Desfazer os X
    for i, bit in enumerate(target_bits):
        if bit == '0':
            oracle.x(i)
    
    return oracle


# Testar o oráculo para o estado |11⟩
print("Oráculo para marcar |11⟩:")
oracle_11 = create_oracle(2, "11")
print(oracle_11.draw())

## 2. Criar o Difusor (Operador de Grover)

O difusor amplifica a amplitude do estado marcado.

In [ ]:
def create_diffuser(n_qubits: int) -> QuantumCircuit:
    """
    Cria o operador difusor de Grover (reflexão sobre a média).
    
    Args:
        n_qubits: número de qubits
    
    Returns:
        QuantumCircuit: difusor
    """
    diffuser = QuantumCircuit(n_qubits)
    
    # Passo 1: Hadamard em todos
    diffuser.h(range(n_qubits))
    
    # Passo 2: X em todos
    diffuser.x(range(n_qubits))
    
    # Passo 3: MCZ (Multi-Controlled Z)
    diffuser.h(n_qubits - 1)
    diffuser.mcx(list(range(n_qubits - 1)), n_qubits - 1)
    diffuser.h(n_qubits - 1)
    
    # Passo 4: X em todos
    diffuser.x(range(n_qubits))
    
    # Passo 5: Hadamard em todos
    diffuser.h(range(n_qubits))
    
    return diffuser


# Testar o difusor
print("Difusor de Grover (2 qubits):")
diff = create_diffuser(2)
print(diff.draw())

## 3. Implementar o Algoritmo Completo

In [ ]:
def grover_search(n_qubits: int, target: str) -> QuantumCircuit:
    """
    Implementa o algoritmo de Grover completo.
    
    Args:
        n_qubits: número de qubits
        target: estado alvo em binário
    
    Returns:
        QuantumCircuit: circuito completo
    """
    # Número ótimo de iterações: floor(π/4 * √N)
    N = 2**n_qubits
    n_iterations = max(1, int(np.floor(np.pi / 4 * np.sqrt(N))))
    
    print(f"N = {N} elementos")
    print(f"Iterações de Grover: {n_iterations}")
    
    # Criar circuito
    qr = QuantumRegister(n_qubits, 'q')
    cr = ClassicalRegister(n_qubits, 'c')
    circuit = QuantumCircuit(qr, cr)
    
    # Passo 1: Superposição inicial
    circuit.h(qr)
    
    # Passo 2: Iterações de Grover
    oracle = create_oracle(n_qubits, target)
    diffuser = create_diffuser(n_qubits)
    
    for i in range(n_iterations):
        circuit.barrier()
        circuit.compose(oracle, inplace=True)
        circuit.compose(diffuser, inplace=True)
    
    # Passo 3: Medir
    circuit.barrier()
    circuit.measure(qr, cr)
    
    return circuit


# Exemplo simples
print("Algoritmo de Grover buscando |11⟩ em 4 elementos:")
grover_2q = grover_search(2, "11")
print(grover_2q.draw())

## 4. Busca em 4 Elementos (2 qubits)

In [ ]:
# Busca em 4 Elementos (2 qubits) - Buscando |11⟩
print("=" * 60)
print("BUSCA EM 4 ELEMENTOS (2 qubits) - Alvo: |11⟩")
print("=" * 60)

target = "11"
circuit = grover_search(2, target)

# Simular
simulator = AerSimulator()
job = simulator.run(circuit, shots=1000)
result = job.result()
counts = result.get_counts()

print(f"\nResultados: {counts}")
print(f"Alvo '{target}' encontrado com probabilidade: {counts.get(target, 0)/1000:.1%}")

# Visualizar
plot_histogram(counts)
plt.title(f"Grover: Busca por |{target}⟩ em 4 elementos")
plt.show()

## 5. Busca em 8 Elementos (3 qubits)

In [ ]:
# Busca em 8 Elementos (3 qubits) - Buscando |101⟩
print("=" * 60)
print("BUSCA EM 8 ELEMENTOS (3 qubits) - Alvo: |101⟩")
print("=" * 60)

target = "101"
circuit = grover_search(3, target)

# Simular
job = simulator.run(circuit, shots=1000)
result = job.result()
counts = result.get_counts()

print(f"\nResultados: {counts}")
print(f"Alvo '{target}' encontrado com probabilidade: {counts.get(target, 0)/1000:.1%}")

# Visualizar
plot_histogram(counts)
plt.title(f"Grover: Busca por |{target}⟩ em 8 elementos")
plt.show()

## 6. Análise de Complexidade

Compare o número de iterações necessárias:

In [ ]:
# TODO: Criar tabela comparando:
# - N elementos
# - Iterações clássicas (N/2 em média)
# - Iterações Grover (√N * π/4)

for n_qubits in range(2, 8):
    N = 2**n_qubits
    classical = N // 2
    quantum = int(np.floor(np.pi / 4 * np.sqrt(N)))
    print(f"N={N:4d} | Clássico: {classical:4d} | Grover: {quantum:2d}")

## 7. Conclusão

Responda:
- Por que o número de iterações importa?
- Para que tamanho de problema o Grover se torna vantajoso?

**Análise:**

1. **Por que o número de iterações importa?**
   - Cada iteração do Grover requer aplicar o oráculo e o difusor
   - Mais iterações = mais operações quânticas = mais ruído em hardware real
   - O número ótimo de iterações é **crítico**: iterações demais diminuem a probabilidade de sucesso

2. **Para que tamanho de problema o Grover se torna vantajoso?**
   - Para N pequeno (< 16), a vantagem é marginal
   - Para N = 1.000.000: Clássico precisa de ~500.000 operações, Grover precisa de ~785
   - O speedup quadrático (√N vs N) se torna significativo em escala

3. **Conexão com o Case 10 (KPMG/TDC Net):**
   - O Grover pode acelerar a busca em espaços de soluções de problemas combinatórios
   - Para TSP/VRP, o espaço de busca cresce fatorialmente (N!)
   - Combinado com QUBO, permite explorar mais soluções em menos tempo